# 08 - Hyperparameter search

Search with Optuna over the hyperparameters in the `search.space` section of `configs/default.yaml`, on
`training.representation` and `training.feature_source`; every other setting comes from the `training` and `model`
sections.

- **Sampler**: TPE, which proposes each trial from the results of the earlier ones.
- **Criterion**: best validation loss of the trial (weighted BCE: rewards both separation and calibration).
- **Pruning**: after `search.pruner.warmup_epochs` epochs, a trial whose validation loss is worse than the median of
  the earlier trials at the same epoch is stopped; pruning starts after `search.pruner.startup_trials` trials.
- **Resumable**: the study is stored in `search.search_dir/<study_name>/study.db`, with one folder per trial
  (checkpoint at every epoch). Run the notebook again after an interruption: the interrupted trial is resumed from
  its last epoch (after `search.grace_period_s` seconds), and the search continues until `search.n_trials` trials are
  finished.

With `search.enqueue_baseline: true` the first trial is the configuration of the `model` and `training` sections
(baseline), trained under the same conditions as the others, as a reference for the gain of the search.

All trials start from the same seed, so they differ only in their hyperparameters. The best configuration is then to be
retrained with 2-3 seeds (notebook 05) before comparing it with others.

## 0. Setup

In [ ]:
from pathlib import Path
import json
import warnings

import matplotlib.pyplot as plt
import optuna
import pandas as pd
from optuna.visualization import matplotlib as optuna_plots

from wqu.config import load_config
from wqu.data.loading import CV_COMPONENTS
from wqu.data.splitting import SPLIT_COLUMN, TRAIN, VALIDATION, attach_split, load_split
from wqu.models.cred import build_model
from wqu.models.loss import bad_class_weight, weighted_bce_loss
from wqu.preprocessing.pipeline import FEATURE_INDEX_COLUMN, feature_file_path, read_feature_records
from wqu.training.data import FeatureDataset, FrequencyStatistics, frequency_statistics, make_loader
from wqu.training.search import (
    BASELINE_ATTRIBUTE,
    enqueue_baseline_trial,
    finished_trials,
    load_study,
    make_objective,
    run_search,
    study_dir,
)
from wqu.training.trainer import select_device
from wqu.visualization import apply_plot_style, save_figure

warnings.filterwarnings("ignore", category=optuna.exceptions.ExperimentalWarning)
optuna.logging.set_verbosity(optuna.logging.WARNING)

CONFIG_PATH = Path("..") / "configs" / "default.yaml"

config = load_config(CONFIG_PATH)
columns = config.columns
training = config.training
search = config.search
apply_plot_style()
device = select_device(training.device)
output_dir = study_dir(search)
output_dir.mkdir(parents=True, exist_ok=True)

features_path = feature_file_path(config.preprocessing, training.feature_source)
records = read_feature_records(features_path, columns.text)
records[SPLIT_COLUMN] = attach_split(records, load_split(config.split.output_file, columns), columns)
train_records = records[records[SPLIT_COLUMN] == TRAIN]
validation_records = records[records[SPLIT_COLUMN] == VALIDATION]
print(f"Study {search.study_name}: {training.representation}, {training.feature_source} "
      f"({len(train_records):,} training / {len(validation_records):,} validation records); device {device}")

## 1. Data

Normalization statistics of the training records (stored in the study folder and reused by every trial and session),
bad-class weight and loaders. Each trial gets a fresh training loader with the same seed.

In [ ]:
normalization_path = output_dir / "normalization.npz"
if normalization_path.is_file():
    statistics = FrequencyStatistics.load(normalization_path)
else:
    statistics = frequency_statistics(features_path, training.representation,
                                      train_records[FEATURE_INDEX_COLUMN].to_numpy())
    statistics.save(normalization_path)
bad_weight = bad_class_weight(train_records[columns.label].to_numpy())


def dataset_for(subset: pd.DataFrame) -> FeatureDataset:
    """Dataset of standardized images for a subset of the records."""
    return FeatureDataset(features_path, training.representation, subset[FEATURE_INDEX_COLUMN].to_numpy(),
                          subset[columns.label].to_numpy(), statistics)


train_dataset = dataset_for(train_records)
validation_loader = make_loader(dataset_for(validation_records), training.batch_size, False, training.num_workers,
                                training.random_seed)
print(f"Bad-class weight: {bad_weight:.3f}")

## 2. Study

The study is created on the first run and loaded afterwards; finished trials are kept.

In [ ]:
study = load_study(search)
if enqueue_baseline_trial(study, search, config.model, training):
    print("Baseline configuration queued as the first trial.")
print(f"Finished trials: {finished_trials(study)} of {search.n_trials}")
pd.Series({
    "learning_rate": f"{search.learning_rate_range} (log)",
    "conv_dropout": search.conv_dropout_choices,
    "recurrent_dropout": search.recurrent_dropout_range,
    "head_dropout": search.head_dropout_range,
    "residual_dropout": search.residual_dropout_options,
    "conv_filters": search.conv_filters_choices,
    "recurrent_units": search.recurrent_units_choices,
}, name="search space").to_frame()

## 3. Search

Runs trials until `search.n_trials` are finished. Each trial prints its epochs; a pruned trial stops with a message.
It can be interrupted at any time and resumed by running the notebook again.

In [ ]:
objective = make_objective(
    search, config.model, training,
    build_model=lambda model_config: build_model(model_config, len(CV_COMPONENTS), len(statistics.mean)),
    loss=weighted_bce_loss(bad_weight),
    make_train_loader=lambda: make_loader(train_dataset, training.batch_size, True, training.num_workers,
                                          training.random_seed),
    validation_loader=validation_loader,
    bad_weight=bad_weight,
    device=device,
)
started = run_search(study, objective, search.n_trials)
print(f"Trials run in this session: {started}; finished trials: {finished_trials(study)}")

## 4. Results

Finished trials sorted by best validation loss (pruned trials have their last reported loss), with the
hyperparameters, the epoch of the best model, AUC and ECE at that epoch, and the duration. Trials interrupted and
resumed under a new number are not listed twice (only the finished copy).

In [ ]:
trials = study.trials_dataframe(attrs=("number", "value", "state", "params", "user_attrs"))
trials = trials[trials["state"].isin(["COMPLETE", "PRUNED"])]
trials["params_conv_filters"] = trials["params_conv_filters"].map(lambda index: list(search.conv_filters_choices[int(index)]))
trials = trials.sort_values("value").rename(columns=lambda name: name.removeprefix("params_").removeprefix("user_attrs_"))
if BASELINE_ATTRIBUTE in trials:
    trials[BASELINE_ATTRIBUTE] = trials[BASELINE_ATTRIBUTE].fillna(False).astype(bool)
trials.to_csv(output_dir / "trials.csv", index=False)
trials.round(4)

In [ ]:
best = study.best_trial
best_settings = {**best.params, "conv_filters": list(search.conv_filters_choices[best.params["conv_filters"]])}
if best_settings["residual_dropout"]:
    best_settings["residual_dropout"] = best_settings["conv_dropout"]
else:
    best_settings["residual_dropout"] = 0.0
(output_dir / "best_settings.json").write_text(json.dumps({"trial": best.number, "validation_loss": best.value,
                                                          **best_settings, **best.user_attrs}, indent=2))
print(f"Best trial: {best.number}, validation loss {best.value:.4f}")
baseline = [trial for trial in study.get_trials(deepcopy=False, states=(optuna.trial.TrialState.COMPLETE,))
            if trial.user_attrs.get(BASELINE_ATTRIBUTE)]
if baseline:
    print(f"Baseline (trial {baseline[0].number}): validation loss {baseline[0].value:.4f}; "
          f"gain of the search {baseline[0].value - best.value:.4f}")
print("Values for configs/default.yaml:")
print(f"  model.conv_filters: {best_settings['conv_filters']}")
print(f"  model.recurrent_units: {best_settings['recurrent_units']}")
for name in ("conv", "recurrent", "head", "residual"):
    print(f"  model.dropout.{name}: {best_settings[f'{name}_dropout']:.3f}")
print(f"  training.learning_rate: {best_settings['learning_rate']:.2e}")

## 5. Figures

Best validation loss per trial and best value so far; validation loss per epoch of every trial (pruned trials end
early); importance of each hyperparameter for the validation loss (fANOVA, needs a few completed trials).

In [ ]:
FIGURES = {
    "optimization_history": optuna_plots.plot_optimization_history,
    "validation_loss_by_epoch": optuna_plots.plot_intermediate_values,
    "hyperparameter_importance": optuna_plots.plot_param_importances,
}
completed = len(study.get_trials(deepcopy=False, states=(optuna.trial.TrialState.COMPLETE,)))
for name, plot in FIGURES.items():
    if name == "hyperparameter_importance" and completed < 3:
        print("Hyperparameter importance needs at least 3 completed trials.")
        continue
    ax = plot(study)
    ax.figure.set_size_inches(9, 5)
    ax.figure.tight_layout()
    save_figure(ax.figure, output_dir / f"{name}.png")
    save_figure(ax.figure, output_dir / f"{name}.pdf")